In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

STAGING_DIR = Path("../data/staging")
PROCESSED_DIR = Path("../data/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

observations_path = STAGING_DIR / "pfas_observations_staging.parquet"
measurements_path = STAGING_DIR / "pfas_measurements_long_staging.parquet"

df_obs = pd.read_parquet(observations_path)
df_meas = pd.read_parquet(measurements_path)

print("Observations staging :", df_obs.shape)
print("Measurements long staging :", df_meas.shape)

display(df_obs.head(2))
display(df_meas.head(2))

Observations staging : (929442, 22)
Measurements long staging : (21530916, 28)


,observation_id,dataset_id,dataset_name,name,country,city,lat,lon,year,date,...,unit,pfas_sum,has_pfas_details,is_positive_pfas_sum,is_zero_pfas_sum,has_valid_coordinates,unit_standardized,matrix_standardized,date_clean,year_clean
0,1,10,15 textile facilities emitting PFAS,Maes,Belgium,Zwevegem,50.808932,3.352552,2018.0,NaN,...,ng/l,130.0,True,True,False,True,ng/L,Surface water,NaT,2018
1,2,10,15 textile facilities emitting PFAS,Tarkett,Belgium,Dendermonde,51.016507,4.088303,2017.0,NaN,...,ng/l,200.0,True,True,False,True,ng/L,Surface water,NaT,2017


,observation_id,dataset_id,dataset_name,name,country,city,lat,lon,year,date,...,pfas_isomer,pfas_less_than,unit_standardized,pfas_unit_standardized,matrix_standardized,date_clean,year_clean,has_valid_coordinates,pfas_less_than_flag,unit_mismatch
0,1,10,15 textile facilities emitting PFAS,Maes,Belgium,Zwevegem,50.808932,3.352552,2018.0,NaN,...,NaN,NaN,ng/L,ng/L,Surface water,NaT,2018,True,False,False
1,1,10,15 textile facilities emitting PFAS,Maes,Belgium,Zwevegem,50.808932,3.352552,2018.0,NaN,...,NaN,NaN,ng/L,ng/L,Surface water,NaT,2018,True,False,False


In [3]:
# Vérifier si observation_id existe dans les deux tables
print("observation_id dans df_obs :", "observation_id" in df_obs.columns)
print("observation_id dans df_meas :", "observation_id" in df_meas.columns)

# Vérifier le nombre d'observations uniques
print("Observations uniques dans df_obs :", df_obs["observation_id"].nunique())
print("Observations uniques dans df_meas :", df_meas["observation_id"].nunique())

observation_id dans df_obs : True
observation_id dans df_meas : True
Observations uniques dans df_obs : 929442
Observations uniques dans df_meas : 929442


In [ ]:
# On travaille à partir de la table observation,
# car une observation correspond à un prélèvement global.

df_date_work = df_obs.copy()

# Vérifier si date_clean existe déjà.
# Si elle n'existe pas, on la crée à partir de la colonne date.
if "date_clean" not in df_date_work.columns:
    df_date_work["date_clean"] = pd.to_datetime(
        df_date_work["date"],
        errors="coerce"
    )

# Vérifier si year_clean existe déjà.
# Si elle n'existe pas, on la crée à partir de year et date_clean.
if "year_clean" not in df_date_work.columns:
    year_from_date = df_date_work["date_clean"].dt.year
    year_from_column = pd.to_numeric(df_date_work["year"], errors="coerce")
    
    df_date_work["year_clean"] = year_from_date.fillna(year_from_column)

# Conversion de year_clean en entier nullable
df_date_work["year_clean"] = df_date_work["year_clean"].astype("Int64")

# Aperçu des colonnes temporelles
display(df_date_work[["date", "date_clean", "year", "year_clean"]].head())


,date,date_clean,year,year_clean
0,NaN,NaT,2018.0,2018
1,NaN,NaT,2017.0,2017
2,NaN,NaT,2016.0,2016
3,NaN,NaT,2017.0,2017
4,NaN,NaT,2018.0,2018


In [6]:
# Fonction pour créer une clé temporelle claire
def build_date_key(row):
    # Si une date complète existe, on utilise la date
    if pd.notna(row["date_clean"]):
        return row["date_clean"].strftime("%Y-%m-%d")
    
    # Si seulement l'année existe, on utilise l'année
    if pd.notna(row["year_clean"]):
        return str(int(row["year_clean"]))
    
    # Si aucune information temporelle n'existe
    return "unknown"


# Création d'une clé temporelle
df_date_work["date_key"] = df_date_work.apply(build_date_key, axis=1)

# Création d'un niveau temporel
df_date_work["date_level"] = np.where(
    df_date_work["date_clean"].notna(),
    "date",
    np.where(df_date_work["year_clean"].notna(), "year_only", "unknown")
)

display(df_date_work[["date_clean", "year_clean", "date_key", "date_level"]].head())

,date_clean,year_clean,date_key,date_level
0,NaT,2018,2018,year_only
1,NaT,2017,2017,year_only
2,NaT,2016,2016,year_only
3,NaT,2017,2017,year_only
4,NaT,2018,2018,year_only


In [7]:
# Sélection des colonnes nécessaires pour la dimension date
dim_date = df_date_work[
    ["date_key", "date_clean", "year_clean", "date_level"]
].drop_duplicates().reset_index(drop=True)

# Création d'une clé technique pour le Data Warehouse
dim_date.insert(0, "date_id", dim_date.index + 1)

# Ajout du mois, du trimestre et du jour quand la date complète existe
dim_date["month"] = dim_date["date_clean"].dt.month
dim_date["quarter"] = dim_date["date_clean"].dt.quarter
dim_date["day"] = dim_date["date_clean"].dt.day

# Réorganisation des colonnes
dim_date = dim_date[
    [
        "date_id",
        "date_key",
        "date_clean",
        "year_clean",
        "month",
        "quarter",
        "day",
        "date_level"
    ]
]

print("Taille de dim_date :", dim_date.shape)

display(dim_date.head())

Taille de dim_date : (5738, 8)


,date_id,date_key,date_clean,year_clean,month,quarter,day,date_level
0,1,2018,NaT,2018,NaN,NaN,NaN,year_only
1,2,2017,NaT,2017,NaN,NaN,NaN,year_only
2,3,2016,NaT,2016,NaN,NaN,NaN,year_only
3,4,2019-07-18,2019-07-18,2019,7.0,3.0,18.0,date
4,5,2019-07-22,2019-07-22,2019,7.0,3.0,22.0,date


In [8]:
# Chemin de sortie de la dimension date
dim_date_output_path = PROCESSED_DIR / "dim_date.parquet"

# Export en Parquet compressé
dim_date.to_parquet(
    dim_date_output_path,
    index=False,
    compression="snappy"
)

print("dim_date exportée vers :", dim_date_output_path)

dim_date exportée vers : ..\data\processed\dim_date.parquet


In [9]:
# Colonnes utiles pour identifier les sources de données
dataset_cols = [
    "dataset_id",
    "dataset_name"
]

# On garde seulement les colonnes qui existent réellement
dataset_cols = [col for col in dataset_cols if col in df_obs.columns]

# Création de la dimension dataset à partir de la table observation
dim_dataset = df_obs[dataset_cols].drop_duplicates().reset_index(drop=True)

# Création d'une clé technique propre au Data Warehouse
dim_dataset.insert(0, "dataset_dw_id", dim_dataset.index + 1)

# Réorganisation des colonnes
dim_dataset = dim_dataset[
    ["dataset_dw_id"] + dataset_cols
]

print("Taille de dim_dataset :", dim_dataset.shape)

display(dim_dataset.head())

Taille de dim_dataset : (107, 3)


,dataset_dw_id,dataset_id,dataset_name
0,1,10,15 textile facilities emitting PFAS
1,2,11,Danube samples
2,3,12,PFAS_in_surface_water_2019-2020
3,4,14,POPMON Trinkwasser Daten 260922
4,5,16,Flanders DOV


In [10]:
# Vérifier le nombre de sources distinctes
print("Nombre de datasets distincts :", dim_dataset.shape[0])

# Vérifier les valeurs manquantes
missing_dataset = pd.DataFrame({
    "column": dim_dataset.columns,
    "missing_values": dim_dataset.isna().sum().values,
    "missing_rate_%": (dim_dataset.isna().mean() * 100).round(2).values
})

display(missing_dataset)

# Vérifier les doublons sur dataset_id si la colonne existe
if "dataset_id" in dim_dataset.columns:
    print("Doublons sur dataset_id :", dim_dataset["dataset_id"].duplicated().sum())

# Aperçu des sources les plus fréquentes dans les observations
display(df_obs["dataset_name"].value_counts(dropna=False).head(20))

Nombre de datasets distincts : 107


,column,missing_values,missing_rate_%
0,dataset_dw_id,0,0.0
1,dataset_id,0,0.0
2,dataset_name,0,0.0


Doublons sur dataset_id : 0


dataset_name
france_naiades                                       328286
france_ades                                          306194
Flanders DOV                                          52223
UK_WIR                                                49534
france_eaurob                                         41320
France - ICPE                                         32135
Water Samplings Veneto                                20876
IT_ARPA_Veneto_PFAS_in_acque_2024_12_04               17265
uk_water_quality                                      10313
DE_NRW_Fliessgewaesser                                 8299
2021-05-19 PFASBodemTotaal                             6381
DE_Baden-Wuerttemberg - Surface water                  6123
IT_Surface_Water_SVG                                   4963
Germany Bayern - Gewässerkundlicher Dienst Bayern      3879
DE_Baden-Wuerttemberg - Groundwater                    3462
denmark_Syddanmark                                     2952
denmark_Midtjyllands       

### Interprétation de `dim_dataset`

La dimension `dim_dataset` regroupe les différentes sources de données présentes dans le dataset PFAS.

Cette dimension est importante car les données proviennent de plusieurs jeux de données, qui peuvent avoir des conventions différentes concernant les unités, les matrices, les valeurs nulles ou les seuils de détection.

Conserver la source des données permet donc d’assurer la traçabilité des observations et d’analyser la qualité ou les comportements spécifiques à chaque dataset.

In [11]:
# Chemin de sortie de la dimension dataset
dim_dataset_output_path = PROCESSED_DIR / "dim_dataset.parquet"

# Export en Parquet compressé
dim_dataset.to_parquet(
    dim_dataset_output_path,
    index=False,
    compression="snappy"
)

print("dim_dataset exportée vers :", dim_dataset_output_path)

dim_dataset exportée vers : ..\data\processed\dim_dataset.parquet


In [ ]:
## 8. Construction de la dimension `dim_matrix`

La dimension `dim_matrix` permet d’analyser les mesures PFAS selon le type de milieu environnemental.

Les PFAS peuvent être mesurés dans plusieurs matrices :
- eaux souterraines ;
- eaux de surface ;
- eaux usées ;
- eau potable ;
- sol ;
- sédiments ;
- biote ;
- lixiviat.

Cette dimension est importante car les concentrations ne sont pas directement comparables entre toutes les matrices.

Le grain de cette dimension est :

Une ligne = une matrice environnementale.

In [12]:
# Vérifier les colonnes liées à la matrice
matrix_columns = [col for col in df_obs.columns if "matrix" in col.lower()]

print("Colonnes liées à la matrice :")
print(matrix_columns)

Colonnes liées à la matrice :
['matrix', 'matrix_standardized']


In [13]:
# Colonnes utiles pour la dimension matrix
matrix_cols = [
    "matrix",
    "matrix_standardized"
]

# Garder seulement les colonnes qui existent
matrix_cols = [col for col in matrix_cols if col in df_obs.columns]

# Création de la dimension matrix
dim_matrix = df_obs[matrix_cols].drop_duplicates().reset_index(drop=True)

# Création d'une clé technique pour le Data Warehouse
dim_matrix.insert(0, "matrix_id", dim_matrix.index + 1)

# Réorganisation des colonnes
dim_matrix = dim_matrix[
    ["matrix_id"] + matrix_cols
]

print("Taille de dim_matrix :", dim_matrix.shape)

display(dim_matrix)

Taille de dim_matrix : (14, 3)


,matrix_id,matrix,matrix_standardized
0,1,Surface water,Surface water
1,2,Groundwater,Groundwater
2,3,Wastewater,Wastewater
3,4,Sediment,Sediment
4,5,Biota,Biota
5,6,Sea water,Sea water
6,7,Drinking water,Drinking water
7,8,Soil,Soil
8,9,Rainwater,Rainwater
9,10,Unknown,Unknown


In [14]:
# Nombre de matrices distinctes
print("Nombre de matrices distinctes :", dim_matrix.shape[0])

# Vérifier les valeurs manquantes
missing_matrix = pd.DataFrame({
    "column": dim_matrix.columns,
    "missing_values": dim_matrix.isna().sum().values,
    "missing_rate_%": (dim_matrix.isna().mean() * 100).round(2).values
})

display(missing_matrix)

# Répartition des observations par matrice standardisée
display(df_obs["matrix_standardized"].value_counts(dropna=False))

Nombre de matrices distinctes : 14


,column,missing_values,missing_rate_%
0,matrix_id,0,0.0
1,matrix,0,0.0
2,matrix_standardized,0,0.0


matrix_standardized
Surface water       449509
Groundwater         394360
Soil                 33796
Sediment             15373
Wastewater           13287
Drinking water       12322
Biota                 5684
Unknown               1886
Suspended matter      1539
Sea water             1334
Rainwater              189
Leachate               149
Sludge                  11
Sewerage                 3
Name: count, dtype: int64

In [ ]:
### Interprétation de `dim_matrix`

La dimension `dim_matrix` regroupe les différents milieux environnementaux présents dans les données PFAS.

Cette dimension est essentielle car les concentrations ne doivent pas être comparées globalement sans tenir compte du milieu analysé. Par exemple, une concentration dans l’eau exprimée en `ng/L` n’est pas directement comparable à une concentration dans le sol exprimée en `ng/kg`.

La standardisation de la matrice permet donc de préparer des analyses plus fiables par type de milieu : eaux souterraines, eaux de surface, eaux usées, sol, sédiments, biote ou autres matrices.

In [15]:
# Chemin de sortie de la dimension matrix
dim_matrix_output_path = PROCESSED_DIR / "dim_matrix.parquet"

# Export en Parquet compressé
dim_matrix.to_parquet(
    dim_matrix_output_path,
    index=False,
    compression="snappy"
)

print("dim_matrix exportée vers :", dim_matrix_output_path)

dim_matrix exportée vers : ..\data\processed\dim_matrix.parquet


## 9. Construction de la dimension `dim_unit`

La dimension `dim_unit` permet d’analyser les mesures PFAS selon l’unité de concentration utilisée.

Cette dimension est importante car les données PFAS mélangent plusieurs unités :
- `ng/L` pour les concentrations dans l’eau ;
- `ng/kg` pour les concentrations dans des matrices solides ;
- `ng/kg dw` pour le poids sec ;
- `ng/kg fw` pour le poids frais.

Les concentrations ne doivent être comparées que lorsque les unités sont cohérentes.

Le grain de cette dimension est :

Une ligne = une unité standardisée.

In [16]:
# Fonction pour classer les unités selon leur type
def classify_unit(unit):
    if pd.isna(unit):
        return "Unknown"
    
    unit = str(unit)
    
    if unit == "ng/L":
        return "Liquid concentration"
    
    if unit == "ng/kg":
        return "Mass concentration"
    
    if unit == "ng/kg dw":
        return "Dry weight concentration"
    
    if unit == "ng/kg fw":
        return "Fresh weight concentration"
    
    if unit == "Unknown":
        return "Unknown"
    
    return "Other"

In [17]:
# Récupérer les unités globales depuis la table observation
units_obs = df_obs[["unit_standardized"]].copy()
units_obs = units_obs.rename(columns={"unit_standardized": "unit_standardized"})

# Récupérer les unités détaillées depuis la table mesure si elles existent
if "pfas_unit_standardized" in df_meas.columns:
    units_meas = df_meas[["pfas_unit_standardized"]].copy()
    units_meas = units_meas.rename(columns={"pfas_unit_standardized": "unit_standardized"})
else:
    units_meas = pd.DataFrame(columns=["unit_standardized"])

# Fusionner toutes les unités utilisées dans le projet
dim_unit = pd.concat(
    [units_obs, units_meas],
    ignore_index=True
)

# Supprimer les doublons
dim_unit = dim_unit.drop_duplicates().reset_index(drop=True)

# Remplacer les valeurs manquantes éventuelles
dim_unit["unit_standardized"] = dim_unit["unit_standardized"].fillna("Unknown")

# Ajouter le type d'unité
dim_unit["unit_type"] = dim_unit["unit_standardized"].apply(classify_unit)

# Créer une clé technique pour le Data Warehouse
dim_unit.insert(0, "unit_id", dim_unit.index + 1)

# Réorganiser les colonnes
dim_unit = dim_unit[
    [
        "unit_id",
        "unit_standardized",
        "unit_type"
    ]
]

print("Taille de dim_unit :", dim_unit.shape)

display(dim_unit)

Taille de dim_unit : (4, 3)


,unit_id,unit_standardized,unit_type
0,1,ng/L,Liquid concentration
1,2,ng/kg,Mass concentration
2,3,ng/kg dw,Dry weight concentration
3,4,ng/kg fw,Fresh weight concentration


In [18]:
# Vérifier le nombre d'unités distinctes
print("Nombre d'unités standardisées :", dim_unit.shape[0])

# Vérifier les valeurs manquantes
missing_unit = pd.DataFrame({
    "column": dim_unit.columns,
    "missing_values": dim_unit.isna().sum().values,
    "missing_rate_%": (dim_unit.isna().mean() * 100).round(2).values
})

display(missing_unit)

# Répartition des unités dans la table observation
print("Unités globales dans df_obs :")
display(df_obs["unit_standardized"].value_counts(dropna=False))

# Répartition des unités détaillées dans la table mesure
if "pfas_unit_standardized" in df_meas.columns:
    print("Unités détaillées dans df_meas :")
    display(df_meas["pfas_unit_standardized"].value_counts(dropna=False))

Nombre d'unités standardisées : 4


,column,missing_values,missing_rate_%
0,unit_id,0,0.0
1,unit_standardized,0,0.0
2,unit_type,0,0.0


Unités globales dans df_obs :


unit_standardized
ng/L        872923
ng/kg dw     40083
ng/kg        11579
ng/kg fw      4857
Name: count, dtype: int64

Unités détaillées dans df_meas :


pfas_unit_standardized
ng/L        20451992
ng/kg dw      840698
ng/kg         174668
ng/kg fw       63558
Name: count, dtype: int64

In [19]:
# Chemin de sortie de la dimension unit
dim_unit_output_path = PROCESSED_DIR / "dim_unit.parquet"

# Export en Parquet compressé
dim_unit.to_parquet(
    dim_unit_output_path,
    index=False,
    compression="snappy"
)

print("dim_unit exportée vers :", dim_unit_output_path)

dim_unit exportée vers : ..\data\processed\dim_unit.parquet


## 11. Construction de la dimension `dim_substance`

La dimension `dim_substance` permet d’analyser les mesures PFAS selon la substance chimique mesurée.

Elle servira à répondre à des questions comme :
- quelles substances PFAS sont les plus fréquentes ?
- quelles substances présentent les concentrations les plus élevées ?
- quelles substances apparaissent dans certaines matrices spécifiques ?
- quelles substances sont associées à des valeurs sous limite de détection ?

Le grain de cette dimension est :

Une ligne = une substance PFAS distincte.

In [20]:
# Colonnes attendues pour décrire les substances PFAS
substance_columns = [
    "pfas_substance",
    "pfas_cas_id",
    "pfas_isomer"
]

# Vérifier quelles colonnes existent dans la table des mesures longues
available_substance_columns = [
    col for col in substance_columns if col in df_meas.columns
]

print("Colonnes disponibles pour dim_substance :")
print(available_substance_columns)

Colonnes disponibles pour dim_substance :
['pfas_substance', 'pfas_cas_id', 'pfas_isomer']


In [22]:
# On travaille à partir de la table détaillée par substance
df_substance_work = df_meas[available_substance_columns].copy()

# Nettoyage du nom de la substance
if "pfas_substance" in df_substance_work.columns:
    df_substance_work["substance_name"] = (
        df_substance_work["pfas_substance"]
        .fillna("Unknown")
        .astype(str)
        .str.strip()
    )
else:
    df_substance_work["substance_name"] = "Unknown"

# Nettoyage du CAS ID
if "pfas_cas_id" in df_substance_work.columns:
    df_substance_work["cas_id"] = (
        df_substance_work["pfas_cas_id"]
        .fillna("Unknown")
        .astype(str)
        .str.strip()
    )
else:
    df_substance_work["cas_id"] = "Unknown"

# Nettoyage de l'isomère
if "pfas_isomer" in df_substance_work.columns:
    df_substance_work["isomer"] = (
        df_substance_work["pfas_isomer"]
        .fillna("Unknown")
        .astype(str)
        .str.strip()
    )
else:
    df_substance_work["isomer"] = "Unknown"

# Création d'un nom standardisé en majuscules
df_substance_work["substance_standardized"] = (
    df_substance_work["substance_name"]
    .str.upper()
)

display(df_substance_work.head())

,pfas_substance,pfas_cas_id,pfas_isomer,substance_name,cas_id,isomer,substance_standardized
0,PFOA,335-67-1,NaN,PFOA,335-67-1,Unknown,PFOA
1,PFOS,1763-23-1,NaN,PFOS,1763-23-1,Unknown,PFOS
2,PFOA,335-67-1,NaN,PFOA,335-67-1,Unknown,PFOA
3,PFOA,335-67-1,NaN,PFOA,335-67-1,Unknown,PFOA
4,PFOS,1763-23-1,NaN,PFOS,1763-23-1,Unknown,PFOS


In [23]:
# Création d'une clé lisible pour identifier une substance
df_substance_work["substance_key"] = (
    df_substance_work["substance_standardized"] + " | " +
    df_substance_work["cas_id"] + " | " +
    df_substance_work["isomer"]
)

display(df_substance_work[[
    "substance_key",
    "substance_name",
    "substance_standardized",
    "cas_id",
    "isomer"
]].head())

,substance_key,substance_name,substance_standardized,cas_id,isomer
0,PFOA | 335-67-1 | Unknown,PFOA,PFOA,335-67-1,Unknown
1,PFOS | 1763-23-1 | Unknown,PFOS,PFOS,1763-23-1,Unknown
2,PFOA | 335-67-1 | Unknown,PFOA,PFOA,335-67-1,Unknown
3,PFOA | 335-67-1 | Unknown,PFOA,PFOA,335-67-1,Unknown
4,PFOS | 1763-23-1 | Unknown,PFOS,PFOS,1763-23-1,Unknown


La clé substance_key permet d’identifier une substance avec plusieurs informations :

nom de la substance + CAS ID + isomère

Cela permet d’éviter de confondre deux substances proches ou deux formes différentes d’une même substance.

In [24]:
# Colonnes à conserver dans la dimension substance
substance_dim_cols = [
    "substance_key",
    "substance_name",
    "substance_standardized",
    "cas_id",
    "isomer"
]

# Création de la dimension substance
dim_substance = (
    df_substance_work[substance_dim_cols]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Création d'une clé technique pour le Data Warehouse
dim_substance.insert(0, "substance_id", dim_substance.index + 1)

print("Taille de dim_substance :", dim_substance.shape)

display(dim_substance.head())

Taille de dim_substance : (295, 6)


,substance_id,substance_key,substance_name,substance_standardized,cas_id,isomer
0,1,PFOA | 335-67-1 | Unknown,PFOA,PFOA,335-67-1,Unknown
1,2,PFOS | 1763-23-1 | Unknown,PFOS,PFOS,1763-23-1,Unknown
2,3,PFHXA | 307-24-4 | Unknown,PFHxA,PFHXA,307-24-4,Unknown
3,4,PFHXS | 355-46-4 | Unknown,PFHxS,PFHXS,355-46-4,Unknown
4,5,PFBS | 375-73-5 | Unknown,PFBS,PFBS,375-73-5,Unknown


In [25]:
# Nombre total de substances distinctes
print("Nombre de substances distinctes :", dim_substance.shape[0])

# Nombre de noms de substances distincts
print("Nombre de noms de substances distincts :")
print(dim_substance["substance_standardized"].nunique())

# Nombre de CAS ID distincts
print("Nombre de CAS ID distincts :")
print(dim_substance["cas_id"].nunique())

# Vérification des valeurs manquantes
missing_substance = pd.DataFrame({
    "column": dim_substance.columns,
    "missing_values": dim_substance.isna().sum().values,
    "missing_rate_%": (dim_substance.isna().mean() * 100).round(2).values
}).sort_values("missing_rate_%", ascending=False)

display(missing_substance)

Nombre de substances distinctes : 295
Nombre de noms de substances distincts :
295
Nombre de CAS ID distincts :
279


,column,missing_values,missing_rate_%
0,substance_id,0,0.0
1,substance_key,0,0.0
2,substance_name,0,0.0
3,substance_standardized,0,0.0
4,cas_id,0,0.0
5,isomer,0,0.0


In [26]:
# Top 20 des substances les plus fréquentes dans la table des mesures longues
top_substances = (
    df_meas["pfas_substance"]
    .value_counts(dropna=False)
    .head(20)
)

display(top_substances)

pfas_substance
Diflufenican          587929
Norflurazon           551757
Tetraconazole         536259
Fludioxonil           529705
Flurochloridone       517012
Flufenacet            506839
Flurtamone            502598
Flazasulfuron         488445
Isoxaflutole          485292
Prosulfuron           473530
Trifloxystrobin       467636
lambda-Cyhalothrin    466974
Benfluralin           463769
Fipronil              450256
Oxyfluorfen           436695
Picoxystrobin         428744
Tefluthrin            426335
Bifenthrin            425597
PFOS                  384857
Tau-Fluvalinate       383304
Name: count, dtype: int64

### Interprétation de `dim_substance`

La dimension `dim_substance` regroupe les substances PFAS présentes dans la table détaillée.

Elle contient le nom de la substance, son identifiant CAS lorsque celui-ci est disponible, ainsi que l’information éventuelle sur l’isomère.

Cette dimension est centrale pour le projet, car elle permettra d’analyser les concentrations PFAS substance par substance, d’identifier les substances les plus fréquentes et de comparer les comportements selon les matrices, les pays ou les sources de données.

In [27]:
# Chemin de sortie de la dimension substance
dim_substance_output_path = PROCESSED_DIR / "dim_substance.parquet"

# Export en Parquet compressé
dim_substance.to_parquet(
    dim_substance_output_path,
    index=False,
    compression="snappy"
)

print("dim_substance exportée vers :", dim_substance_output_path)

dim_substance exportée vers : ..\data\processed\dim_substance.parquet


## 10. Construction de la dimension `dim_location`

La dimension `dim_location` permet d’analyser les mesures PFAS selon leur localisation géographique.

Elle servira à répondre à des questions comme :
- quels pays contiennent le plus de mesures PFAS ?
- quels sites présentent les concentrations les plus élevées ?
- quelles zones géographiques doivent être surveillées en priorité ?
- quelles observations peuvent être affichées sur une carte ?

Le grain de cette dimension est :

Une ligne = une localisation ou un site de mesure.

In [33]:
# Colonnes géographiques attendues
location_columns = ["name", "country", "city", "lat", "lon", "has_valid_coordinates"]

# Vérifier quelles colonnes existent dans df_obs
available_location_columns = [col for col in location_columns if col in df_obs.columns]

print("Colonnes géographiques disponibles :")
print(available_location_columns)

Colonnes géographiques disponibles :
['name', 'country', 'city', 'lat', 'lon', 'has_valid_coordinates']


In [34]:
# On travaille à partir de la table observation
# car la localisation est liée à une observation globale
df_location_work = df_obs.copy()

# Nettoyer les colonnes textuelles
df_location_work["name_clean"] = df_location_work["name"].fillna("Unknown").astype(str).str.strip()
df_location_work["country_clean"] = df_location_work["country"].fillna("Unknown").astype(str).str.strip()
df_location_work["city_clean"] = df_location_work["city"].fillna("Unknown").astype(str).str.strip()

# Convertir les coordonnées en valeurs numériques
df_location_work["lat_clean"] = pd.to_numeric(df_location_work["lat"], errors="coerce")
df_location_work["lon_clean"] = pd.to_numeric(df_location_work["lon"], errors="coerce")

# Vérifier la validité des coordonnées
df_location_work["has_valid_coordinates"] = (
    df_location_work["lat_clean"].between(-90, 90) &
    df_location_work["lon_clean"].between(-180, 180)
)

# Arrondir les coordonnées pour éviter de créer des doublons artificiels
df_location_work["lat_rounded"] = df_location_work["lat_clean"].round(6)
df_location_work["lon_rounded"] = df_location_work["lon_clean"].round(6)

display(df_location_work[
    ["name_clean", "country_clean", "city_clean", "lat_clean", "lon_clean", "has_valid_coordinates"]
].head())

,name_clean,country_clean,city_clean,lat_clean,lon_clean,has_valid_coordinates
0,Maes,Belgium,Zwevegem,50.808932,3.352552,True
1,Tarkett,Belgium,Dendermonde,51.016507,4.088303,True
2,Ververijen Escotex,Belgium,Deinze,51.042282,3.548967,True
3,Gerhard Van Clewe,Germany,Hamminkeln-Dingden,51.771554,6.605953,True
4,KOB Karl Otto Braun,Germany,Wolfstein,49.590101,7.603395,True


In [35]:
# Création d'une clé lisible pour identifier une localisation
df_location_work["location_key"] = (
    df_location_work["country_clean"] + " | " +
    df_location_work["city_clean"] + " | " +
    df_location_work["name_clean"] + " | " +
    df_location_work["lat_rounded"].astype(str) + " | " +
    df_location_work["lon_rounded"].astype(str)
)

display(df_location_work[["location_key"]].head())

,location_key
0,Belgium | Zwevegem | Maes | 50.808932 | 3.352552
1,Belgium | Dendermonde | Tarkett | 51.016507 | ...
2,Belgium | Deinze | Ververijen Escotex | 51.042...
3,Germany | Hamminkeln-Dingden | Gerhard Van Cle...
4,Germany | Wolfstein | KOB Karl Otto Braun | 49...


In [36]:
# Colonnes à conserver dans la dimension localisation
location_dim_cols = [
    "location_key",
    "name_clean",
    "country_clean",
    "city_clean",
    "lat_clean",
    "lon_clean",
    "has_valid_coordinates"
]

# Création de la dimension location
dim_location = df_location_work[location_dim_cols].drop_duplicates().reset_index(drop=True)

# Création d'une clé technique pour le Data Warehouse
dim_location.insert(0, "location_id", dim_location.index + 1)

# Renommer les colonnes pour un format plus clair
dim_location = dim_location.rename(columns={
    "name_clean": "site_name",
    "country_clean": "country",
    "city_clean": "city",
    "lat_clean": "lat",
    "lon_clean": "lon"
})

print("Taille de dim_location :", dim_location.shape)

display(dim_location.head())

Taille de dim_location : (115909, 8)


,location_id,location_key,site_name,country,city,lat,lon,has_valid_coordinates
0,1,Belgium | Zwevegem | Maes | 50.808932 | 3.352552,Maes,Belgium,Zwevegem,50.808932,3.352552,True
1,2,Belgium | Dendermonde | Tarkett | 51.016507 | ...,Tarkett,Belgium,Dendermonde,51.016507,4.088303,True
2,3,Belgium | Deinze | Ververijen Escotex | 51.042...,Ververijen Escotex,Belgium,Deinze,51.042282,3.548967,True
3,4,Germany | Hamminkeln-Dingden | Gerhard Van Cle...,Gerhard Van Clewe,Germany,Hamminkeln-Dingden,51.771554,6.605953,True
4,5,Germany | Wolfstein | KOB Karl Otto Braun | 49...,KOB Karl Otto Braun,Germany,Wolfstein,49.590101,7.603395,True


In [37]:
# Nombre total de localisations distinctes
print("Nombre de localisations distinctes :", dim_location.shape[0])

# Nombre de pays distincts
print("Nombre de pays distincts :", dim_location["country"].nunique())

# Taux de coordonnées valides
valid_coordinates_rate = dim_location["has_valid_coordinates"].mean() * 100
print(f"Taux de coordonnées valides : {valid_coordinates_rate:.2f}%")

# Top 20 des pays les plus représentés
display(dim_location["country"].value_counts(dropna=False).head(20))

Nombre de localisations distinctes : 115909
Nombre de pays distincts : 34
Taux de coordonnées valides : 89.83%


country
France            56419
Belgium           30207
Italy              6627
Germany            5205
Denmark            4739
United Kingdom     4545
Netherlands        4497
Unknown            1314
Sweden              579
Spain               344
Finland             306
Switzerland         273
Norway              134
Lithuania           132
Latvia              113
Czechia              96
Austria              61
Malta                49
Hungary              34
Serbia               31
Name: count, dtype: int64

In [38]:
# Analyse des valeurs manquantes dans dim_location
missing_location = pd.DataFrame({
    "column": dim_location.columns,
    "missing_values": dim_location.isna().sum().values,
    "missing_rate_%": (dim_location.isna().mean() * 100).round(2).values
}).sort_values("missing_rate_%", ascending=False)

display(missing_location)

,column,missing_values,missing_rate_%
1,location_key,11787,10.17
5,lat,11787,10.17
6,lon,11787,10.17
0,location_id,0,0.00
3,country,0,0.00
2,site_name,0,0.00
4,city,0,0.00
7,has_valid_coordinates,0,0.00


### Interprétation de `dim_location`

La dimension `dim_location` regroupe les sites et localisations présents dans les données PFAS.

Elle permet de conserver les informations géographiques nécessaires aux futures analyses :
- pays ;
- ville ;
- nom du site ;
- latitude ;
- longitude ;
- validité des coordonnées.

Cette dimension sera utilisée dans le Data Warehouse pour analyser les observations par territoire et pour construire des visualisations cartographiques dans Metabase.

La création d’une clé `location_key` permet d’éviter de considérer deux sites différents comme identiques lorsqu’ils ont le même nom mais des localisations différentes.

In [39]:
# Chemin de sortie de la dimension location
dim_location_output_path = PROCESSED_DIR / "dim_location.parquet"

# Export en Parquet compressé
dim_location.to_parquet(
    dim_location_output_path,
    index=False,
    compression="snappy"
)

print("dim_location exportée vers :", dim_location_output_path)

dim_location exportée vers : ..\data\processed\dim_location.parquet


In [45]:
# Vérifier si location_key est unique dans dim_location
print("Taille dim_location :", dim_location.shape)
print("Nombre de location_key uniques :", dim_location["location_key"].nunique())
print("Nombre de doublons location_key :", dim_location["location_key"].duplicated().sum())

# Voir les clés de localisation les plus répétées
display(dim_location["location_key"].value_counts().head(10))

Taille dim_location : (115909, 8)
Nombre de location_key uniques : 104122
Nombre de doublons location_key : 11786


location_key
Belgium | Zwevegem | Maes | 50.808932 | 3.352552                                1
Belgium | Dendermonde | Tarkett | 51.016507 | 4.088303                          1
Belgium | Deinze | Ververijen Escotex | 51.042282 | 3.548967                    1
Germany | Hamminkeln-Dingden | Gerhard Van Clewe | 51.771554 | 6.605953         1
Germany | Wolfstein | KOB Karl Otto Braun | 49.590101 | 7.603395                1
Germany | Schüttorf | Rofa | 52.319134 | 7.228233                              1
Italy | Tollegno | Tollegno 1900 | 45.585137 | 8.053703                         1
Poland | Vila Nova de Famalicão | Faria & Coelho | 41.407306 | -8.387184        1
Sweden | Kinna | Almedahl-Kinna | 57.503802 | 12.698008                         1
Slovakia | Šamorín | Samorin Kalinkovo_Samorin_Slovakia | 48.02977 | 17.2838    1
Name: count, dtype: int64

In [46]:
# Garder une seule ligne par location_key
dim_location = (
    dim_location
    .drop_duplicates(subset=["location_key"])
    .reset_index(drop=True)
)

# Recréer location_id proprement
dim_location = dim_location.drop(columns=["location_id"], errors="ignore")
dim_location.insert(0, "location_id", dim_location.index + 1)

print("Nouvelle taille dim_location :", dim_location.shape)
print("Doublons location_key :", dim_location["location_key"].duplicated().sum())

display(dim_location.head())

Nouvelle taille dim_location : (104123, 8)
Doublons location_key : 0


,location_id,location_key,site_name,country,city,lat,lon,has_valid_coordinates
0,1,Belgium | Zwevegem | Maes | 50.808932 | 3.352552,Maes,Belgium,Zwevegem,50.808932,3.352552,True
1,2,Belgium | Dendermonde | Tarkett | 51.016507 | ...,Tarkett,Belgium,Dendermonde,51.016507,4.088303,True
2,3,Belgium | Deinze | Ververijen Escotex | 51.042...,Ververijen Escotex,Belgium,Deinze,51.042282,3.548967,True
3,4,Germany | Hamminkeln-Dingden | Gerhard Van Cle...,Gerhard Van Clewe,Germany,Hamminkeln-Dingden,51.771554,6.605953,True
4,5,Germany | Wolfstein | KOB Karl Otto Braun | 49...,KOB Karl Otto Braun,Germany,Wolfstein,49.590101,7.603395,True


In [47]:
dim_location_output_path = PROCESSED_DIR / "dim_location.parquet"

dim_location.to_parquet(
    dim_location_output_path,
    index=False,
    compression="snappy"
)

print("dim_location corrigée exportée vers :", dim_location_output_path)

dim_location corrigée exportée vers : ..\data\processed\dim_location.parquet


In [ ]:
## 12. Construction de la table de faits `fact_pfas_observation`

La table `fact_pfas_observation` représente le niveau global d’une observation PFAS.
Le grain de cette table est :
Une ligne = une observation environnementale globale.

Cette table contient les mesures globales comme :
- `pfas_sum` ;
- l’indicateur de valeur positive ;
- l’indicateur de valeur nulle ;
- l’indicateur de valeur extrême ;
- l’indicateur de coordonnées valides.

Elle contient aussi les clés étrangères vers les dimensions :
- `dim_date` ;
- `dim_location` ;
- `dim_dataset` ;
- `dim_matrix` ;
- `dim_unit`.

Cette table servira à analyser la contamination globale par site, pays, année, matrice, unité ou source de données.

In [51]:
import gc

# On travaille avec une copie légère de df_obs
fact_obs_work = df_obs.copy()

print("Taille initiale :", fact_obs_work.shape)

Taille initiale : (929442, 22)


In [52]:
# Sécuriser date_clean
if "date_clean" not in fact_obs_work.columns:
    fact_obs_work["date_clean"] = pd.to_datetime(
        fact_obs_work["date"],
        errors="coerce"
    )

# Sécuriser year_clean
if "year_clean" not in fact_obs_work.columns:
    year_from_date = fact_obs_work["date_clean"].dt.year
    year_from_column = pd.to_numeric(fact_obs_work["year"], errors="coerce")
    fact_obs_work["year_clean"] = year_from_date.fillna(year_from_column)

fact_obs_work["year_clean"] = fact_obs_work["year_clean"].astype("Int64")


# Construire la même date_key que dans dim_date
def build_date_key(row):
    if pd.notna(row["date_clean"]):
        return row["date_clean"].strftime("%Y-%m-%d")
    
    if pd.notna(row["year_clean"]):
        return str(int(row["year_clean"]))
    
    return "unknown"


fact_obs_work["date_key"] = fact_obs_work.apply(build_date_key, axis=1)

In [53]:
# Nettoyage des champs géographiques
fact_obs_work["name_clean"] = fact_obs_work["name"].fillna("Unknown").astype(str).str.strip()
fact_obs_work["country_clean"] = fact_obs_work["country"].fillna("Unknown").astype(str).str.strip()
fact_obs_work["city_clean"] = fact_obs_work["city"].fillna("Unknown").astype(str).str.strip()

# Coordonnées numériques
fact_obs_work["lat_clean"] = pd.to_numeric(fact_obs_work["lat"], errors="coerce")
fact_obs_work["lon_clean"] = pd.to_numeric(fact_obs_work["lon"], errors="coerce")

# Même arrondi que dans dim_location
fact_obs_work["lat_rounded"] = fact_obs_work["lat_clean"].round(6)
fact_obs_work["lon_rounded"] = fact_obs_work["lon_clean"].round(6)

# Même clé que dans dim_location
fact_obs_work["location_key"] = (
    fact_obs_work["country_clean"] + " | " +
    fact_obs_work["city_clean"] + " | " +
    fact_obs_work["name_clean"] + " | " +
    fact_obs_work["lat_rounded"].astype(str) + " | " +
    fact_obs_work["lon_rounded"].astype(str)
)

In [54]:
# Sécuriser l'unicité des clés dans les dimensions
dim_date_unique = dim_date.drop_duplicates(subset=["date_key"])
dim_location_unique = dim_location.drop_duplicates(subset=["location_key"])
dim_dataset_unique = dim_dataset.drop_duplicates(subset=["dataset_id", "dataset_name"])
dim_matrix_unique = dim_matrix.drop_duplicates(subset=["matrix", "matrix_standardized"])
dim_unit_unique = dim_unit.drop_duplicates(subset=["unit_standardized"])


# 1. Dictionnaire date_key → date_id
date_map = dim_date_unique.set_index("date_key")["date_id"]
fact_obs_work["date_id"] = fact_obs_work["date_key"].map(date_map)


# 2. Dictionnaire location_key → location_id
location_map = dim_location_unique.set_index("location_key")["location_id"]
fact_obs_work["location_id"] = fact_obs_work["location_key"].map(location_map)


# 3. Dictionnaire dataset_key → dataset_dw_id
dim_dataset_unique["dataset_key"] = (
    dim_dataset_unique["dataset_id"].astype(str) + " | " +
    dim_dataset_unique["dataset_name"].astype(str)
)

fact_obs_work["dataset_key"] = (
    fact_obs_work["dataset_id"].astype(str) + " | " +
    fact_obs_work["dataset_name"].astype(str)
)

dataset_map = dim_dataset_unique.set_index("dataset_key")["dataset_dw_id"]
fact_obs_work["dataset_dw_id"] = fact_obs_work["dataset_key"].map(dataset_map)


# 4. Dictionnaire matrix_key → matrix_id
dim_matrix_unique["matrix_key"] = (
    dim_matrix_unique["matrix"].astype(str) + " | " +
    dim_matrix_unique["matrix_standardized"].astype(str)
)

fact_obs_work["matrix_key"] = (
    fact_obs_work["matrix"].astype(str) + " | " +
    fact_obs_work["matrix_standardized"].astype(str)
)

matrix_map = dim_matrix_unique.set_index("matrix_key")["matrix_id"]
fact_obs_work["matrix_id"] = fact_obs_work["matrix_key"].map(matrix_map)


# 5. Dictionnaire unit_standardized → unit_id
unit_map = dim_unit_unique.set_index("unit_standardized")["unit_id"]
fact_obs_work["unit_id"] = fact_obs_work["unit_standardized"].map(unit_map)


print("Clés de dimensions ajoutées")
display(fact_obs_work[[
    "observation_id",
    "date_id",
    "location_id",
    "dataset_dw_id",
    "matrix_id",
    "unit_id"
]].head())

Clés de dimensions ajoutées


,observation_id,date_id,location_id,dataset_dw_id,matrix_id,unit_id
0,1,1,1,1,1,1
1,2,2,2,1,1,1
2,3,3,3,1,1,1
3,4,2,4,1,1,1
4,5,1,5,1,1,1


In [55]:
foreign_keys = [
    "date_id",
    "location_id",
    "dataset_dw_id",
    "matrix_id",
    "unit_id"
]

fk_quality = pd.DataFrame({
    "foreign_key": foreign_keys,
    "missing_values": [fact_obs_work[col].isna().sum() for col in foreign_keys],
    "missing_rate_%": [
        round(fact_obs_work[col].isna().mean() * 100, 2)
        for col in foreign_keys
    ]
})

display(fk_quality)

,foreign_key,missing_values,missing_rate_%
0,date_id,0,0.0
1,location_id,0,0.0
2,dataset_dw_id,0,0.0
3,matrix_id,0,0.0
4,unit_id,0,0.0


In [56]:
fact_obs_cols = [
    "observation_id",
    "date_id",
    "location_id",
    "dataset_dw_id",
    "matrix_id",
    "unit_id",
    "pfas_sum",
    "is_missing_pfas_sum",
    "is_zero_pfas_sum",
    "is_positive_pfas_sum",
    "is_extreme_pfas_sum",
    "has_valid_coordinates"
]

# Garder seulement les colonnes disponibles
fact_obs_cols = [col for col in fact_obs_cols if col in fact_obs_work.columns]

# Créer la table de faits finale
fact_pfas_observation = fact_obs_work[fact_obs_cols].copy()

# Ajouter une clé technique
fact_pfas_observation.insert(
    0,
    "fact_observation_id",
    range(1, len(fact_pfas_observation) + 1)
)

print("Taille de fact_pfas_observation :", fact_pfas_observation.shape)

display(fact_pfas_observation.head())

Taille de fact_pfas_observation : (929442, 11)


,fact_observation_id,observation_id,date_id,location_id,dataset_dw_id,matrix_id,unit_id,pfas_sum,is_zero_pfas_sum,is_positive_pfas_sum,has_valid_coordinates
0,1,1,1,1,1,1,1,130.0,False,True,True
1,2,2,2,2,1,1,1,200.0,False,True,True
2,3,3,3,3,1,1,1,42400.0,False,True,True
3,4,4,2,4,1,1,1,50.0,False,True,True
4,5,5,1,5,1,1,1,580.0,False,True,True


In [57]:
fact_obs_output_path = PROCESSED_DIR / "fact_pfas_observation.parquet"

fact_pfas_observation.to_parquet(
    fact_obs_output_path,
    index=False,
    compression="snappy"
)

print("fact_pfas_observation exportée vers :", fact_obs_output_path)

fact_pfas_observation exportée vers : ..\data\processed\fact_pfas_observation.parquet


## 13. Construction de la table de faits `fact_pfas_measurement`

La table `fact_pfas_measurement` représente le niveau détaillé des mesures PFAS.

Le grain de cette table est :

Une ligne = une substance PFAS mesurée dans une observation.

Cette table permet d’analyser les concentrations PFAS substance par substance.

Elle contient :
- la valeur individuelle `pfas_value` ;
- l’indicateur `pfas_less_than_flag` ;
- les indicateurs qualité sur la valeur individuelle ;
- les clés vers les dimensions ;
- l’identifiant de l’observation d’origine.

Cette table servira à répondre à des questions comme :
- quelles substances PFAS sont les plus fréquentes ?
- quelles substances ont les concentrations les plus élevées ?
- quelles substances apparaissent dans certaines matrices ou certains pays ?
- quelles mesures sont positives, nulles ou sous limite de détection ?

In [58]:
import gc

# Colonnes utiles pour construire la table de faits détaillée
measurement_cols = [
    "observation_id",
    "pfas_value",
    "pfas_less_than_flag",
    "is_missing_pfas_value",
    "is_zero_pfas_value",
    "is_positive_pfas_value",
    "pfas_substance",
    "pfas_cas_id",
    "pfas_isomer",
    "pfas_unit",
    "pfas_unit_standardized"
]

# Garder seulement les colonnes qui existent réellement
measurement_cols = [col for col in measurement_cols if col in df_meas.columns]

# Création d'une table de travail légère
fact_meas_work = df_meas[measurement_cols].copy()

print("Taille initiale de fact_meas_work :", fact_meas_work.shape)

display(fact_meas_work.head())

Taille initiale de fact_meas_work : (21530916, 8)


,observation_id,pfas_value,pfas_less_than_flag,pfas_substance,pfas_cas_id,pfas_isomer,pfas_unit,pfas_unit_standardized
0,1,90.0,False,PFOA,335-67-1,NaN,ng/l,ng/L
1,1,40.0,False,PFOS,1763-23-1,NaN,ng/l,ng/L
2,2,200.0,False,PFOA,335-67-1,NaN,ng/l,ng/L
3,3,41400.0,False,PFOA,335-67-1,NaN,ng/l,ng/L
4,3,500.0,False,PFOS,1763-23-1,NaN,ng/l,ng/L


### Ajout des clés communes aux observations

La table `fact_pfas_measurement` est liée à la table `fact_pfas_observation` par `observation_id`.

Comme chaque mesure détaillée appartient à une observation, nous pouvons récupérer les clés des dimensions déjà calculées dans `fact_pfas_observation` :
- `date_id` ;
- `location_id` ;
- `dataset_dw_id` ;
- `matrix_id` ;
- `unit_id`.

Cela évite de refaire les jointures avec toutes les dimensions et réduit le risque d’erreur mémoire.

In [59]:
# Si fact_pfas_observation n'est plus en mémoire, on la recharge
fact_obs_path = PROCESSED_DIR / "fact_pfas_observation.parquet"

if "fact_pfas_observation" not in globals():
    fact_pfas_observation = pd.read_parquet(fact_obs_path)

# Colonnes de clés à récupérer
obs_key_cols = [
    "observation_id",
    "date_id",
    "location_id",
    "dataset_dw_id",
    "matrix_id",
    "unit_id"
]

# Création d'une table lookup : observation_id -> clés des dimensions
obs_lookup = (
    fact_pfas_observation[obs_key_cols]
    .drop_duplicates(subset=["observation_id"])
    .set_index("observation_id")
)

# Ajout des clés dans la table de mesures détaillées
for col in ["date_id", "location_id", "dataset_dw_id", "matrix_id", "unit_id"]:
    fact_meas_work[col] = fact_meas_work["observation_id"].map(obs_lookup[col])

print("Clés d'observation ajoutées")

display(fact_meas_work[
    ["observation_id", "date_id", "location_id", "dataset_dw_id", "matrix_id", "unit_id"]
].head())

Clés d'observation ajoutées


,observation_id,date_id,location_id,dataset_dw_id,matrix_id,unit_id
0,1,1,1,1,1,1
1,1,1,1,1,1,1
2,2,2,2,1,1,1
3,3,3,3,1,1,1
4,3,3,3,1,1,1


In [60]:
# Conversion de pfas_value en numérique
fact_meas_work["pfas_value"] = pd.to_numeric(
    fact_meas_work["pfas_value"],
    errors="coerce"
)

# Créer les indicateurs si jamais ils n'existent pas déjà
if "is_missing_pfas_value" not in fact_meas_work.columns:
    fact_meas_work["is_missing_pfas_value"] = fact_meas_work["pfas_value"].isna()

if "is_zero_pfas_value" not in fact_meas_work.columns:
    fact_meas_work["is_zero_pfas_value"] = fact_meas_work["pfas_value"] == 0

if "is_positive_pfas_value" not in fact_meas_work.columns:
    fact_meas_work["is_positive_pfas_value"] = fact_meas_work["pfas_value"] > 0

display(fact_meas_work["pfas_value"].describe())

count    1.165797e+06
mean     1.362051e+04
std      9.055518e+05
min      2.000000e-04
25%      3.000000e+00
50%      1.000000e+01
75%      5.363000e+01
max      5.090000e+08
Name: pfas_value, dtype: float64

### Ajout de la clé `substance_id`

Chaque ligne de `fact_pfas_measurement` correspond à une substance PFAS.

Nous devons donc relier chaque mesure à la dimension `dim_substance`.

Pour cela, nous reconstruisons la même clé logique que celle utilisée dans `dim_substance` :

substance standardisée + CAS ID + isomère

Puis nous récupérons l’identifiant technique `substance_id`.

In [61]:
# Nettoyage du nom de substance
substance_name = (
    fact_meas_work["pfas_substance"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
)

# Nettoyage du CAS ID
cas_id = (
    fact_meas_work["pfas_cas_id"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
)

# Nettoyage de l'isomère
isomer = (
    fact_meas_work["pfas_isomer"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
)

# Création de la même clé que dans dim_substance
fact_meas_work["substance_key"] = (
    substance_name.str.upper() + " | " + cas_id + " | " + isomer
)

# Dictionnaire substance_key -> substance_id
substance_map = (
    dim_substance
    .drop_duplicates(subset=["substance_key"])
    .set_index("substance_key")["substance_id"]
)

# Ajout de substance_id
fact_meas_work["substance_id"] = fact_meas_work["substance_key"].map(substance_map)

# Suppression de la clé temporaire pour économiser la mémoire
fact_meas_work = fact_meas_work.drop(columns=["substance_key"])

gc.collect()

print("substance_id ajouté")

display(fact_meas_work[["pfas_substance", "pfas_cas_id", "pfas_isomer", "substance_id"]].head())

substance_id ajouté


,pfas_substance,pfas_cas_id,pfas_isomer,substance_id
0,PFOA,335-67-1,NaN,1
1,PFOS,1763-23-1,NaN,2
2,PFOA,335-67-1,NaN,1
3,PFOA,335-67-1,NaN,1
4,PFOS,1763-23-1,NaN,2


### Ajout de l’unité détaillée PFAS

Dans `fact_pfas_measurement`, la valeur principale est `pfas_value`.

Son unité correspond à `pfas_unit_standardized`.

Nous ajoutons donc une clé `pfas_unit_id`, qui pointe vers `dim_unit`.

In [62]:
# Si pfas_unit_standardized n'existe pas, on la crée depuis pfas_unit
if "pfas_unit_standardized" not in fact_meas_work.columns:
    fact_meas_work["pfas_unit_standardized"] = fact_meas_work["pfas_unit"].apply(standardize_unit)

# Dictionnaire unit_standardized -> unit_id
unit_map = (
    dim_unit
    .drop_duplicates(subset=["unit_standardized"])
    .set_index("unit_standardized")["unit_id"]
)

# Unité propre à la mesure individuelle
fact_meas_work["pfas_unit_id"] = fact_meas_work["pfas_unit_standardized"].map(unit_map)

display(fact_meas_work[["pfas_unit", "pfas_unit_standardized", "pfas_unit_id"]].head())

,pfas_unit,pfas_unit_standardized,pfas_unit_id
0,ng/l,ng/L,1
1,ng/l,ng/L,1
2,ng/l,ng/L,1
3,ng/l,ng/L,1
4,ng/l,ng/L,1


In [63]:
# Liste des clés étrangères attendues
measurement_foreign_keys = [
    "date_id",
    "location_id",
    "dataset_dw_id",
    "matrix_id",
    "unit_id",
    "pfas_unit_id",
    "substance_id"
]

# Résumé des valeurs manquantes dans les clés
fk_quality_meas = pd.DataFrame({
    "foreign_key": measurement_foreign_keys,
    "missing_values": [fact_meas_work[col].isna().sum() for col in measurement_foreign_keys],
    "missing_rate_%": [
        round(fact_meas_work[col].isna().mean() * 100, 2)
        for col in measurement_foreign_keys
    ]
})

display(fk_quality_meas)

,foreign_key,missing_values,missing_rate_%
0,date_id,0,0.0
1,location_id,0,0.0
2,dataset_dw_id,0,0.0
3,matrix_id,0,0.0
4,unit_id,0,0.0
5,pfas_unit_id,0,0.0
6,substance_id,0,0.0


In [64]:
# Colonnes finales de la table de faits détaillée
fact_measurement_cols = [
    "observation_id",
    "date_id",
    "location_id",
    "dataset_dw_id",
    "matrix_id",
    "unit_id",
    "pfas_unit_id",
    "substance_id",
    "pfas_value",
    "pfas_less_than_flag",
    "is_missing_pfas_value",
    "is_zero_pfas_value",
    "is_positive_pfas_value"
]

# Garder seulement les colonnes disponibles
fact_measurement_cols = [col for col in fact_measurement_cols if col in fact_meas_work.columns]

# Création de la table finale
fact_pfas_measurement = fact_meas_work[fact_measurement_cols].copy()

# Ajout d'une clé technique pour la table de faits
fact_pfas_measurement.insert(
    0,
    "fact_measurement_id",
    range(1, len(fact_pfas_measurement) + 1)
)

print("Taille de fact_pfas_measurement :", fact_pfas_measurement.shape)

display(fact_pfas_measurement.head())

Taille de fact_pfas_measurement : (21530916, 14)


,fact_measurement_id,observation_id,date_id,location_id,dataset_dw_id,matrix_id,unit_id,pfas_unit_id,substance_id,pfas_value,pfas_less_than_flag,is_missing_pfas_value,is_zero_pfas_value,is_positive_pfas_value
0,1,1,1,1,1,1,1,1,1,90.0,False,False,False,True
1,2,1,1,1,1,1,1,1,2,40.0,False,False,False,True
2,3,2,2,2,1,1,1,1,1,200.0,False,False,False,True
3,4,3,3,3,1,1,1,1,1,41400.0,False,False,False,True
4,5,3,3,3,1,1,1,1,2,500.0,False,False,False,True


In [65]:
# Résumé des valeurs manquantes
missing_fact_meas = pd.DataFrame({
    "column": fact_pfas_measurement.columns,
    "missing_values": fact_pfas_measurement.isna().sum().values,
    "missing_rate_%": (fact_pfas_measurement.isna().mean() * 100).round(2).values
}).sort_values("missing_rate_%", ascending=False)

display(missing_fact_meas)

# Quelques indicateurs rapides
print("Nombre total de mesures détaillées :", fact_pfas_measurement.shape[0])

if "is_positive_pfas_value" in fact_pfas_measurement.columns:
    positive_rate = fact_pfas_measurement["is_positive_pfas_value"].mean() * 100
    print(f"Taux de mesures individuelles positives : {positive_rate:.2f}%")

if "is_zero_pfas_value" in fact_pfas_measurement.columns:
    zero_rate = fact_pfas_measurement["is_zero_pfas_value"].mean() * 100
    print(f"Taux de mesures individuelles nulles : {zero_rate:.2f}%")

if "pfas_less_than_flag" in fact_pfas_measurement.columns:
    less_than_rate = fact_pfas_measurement["pfas_less_than_flag"].mean() * 100
    print(f"Taux de mesures sous limite : {less_than_rate:.2f}%")

,column,missing_values,missing_rate_%
9,pfas_value,20365119,94.59
0,fact_measurement_id,0,0.00
2,date_id,0,0.00
3,location_id,0,0.00
4,dataset_dw_id,0,0.00
1,observation_id,0,0.00
5,matrix_id,0,0.00
6,unit_id,0,0.00
7,pfas_unit_id,0,0.00
8,substance_id,0,0.00


Nombre total de mesures détaillées : 21530916
Taux de mesures individuelles positives : 5.41%
Taux de mesures individuelles nulles : 0.00%
Taux de mesures sous limite : 0.00%


In [66]:
# Chemin de sortie de la table de faits détaillée
fact_meas_output_path = PROCESSED_DIR / "fact_pfas_measurement.parquet"

# Export en Parquet compressé
fact_pfas_measurement.to_parquet(
    fact_meas_output_path,
    index=False,
    compression="snappy"
)

print("fact_pfas_measurement exportée vers :", fact_meas_output_path)

fact_pfas_measurement exportée vers : ..\data\processed\fact_pfas_measurement.parquet
